<a href="https://colab.research.google.com/github/jnlyu6/Diehl_Cook_2015_Replication/blob/main/Diehl_Cook_2015.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ***Attention***
*   To facilitate seamless execution, the open-source implementation of the Diehl-Cook SNN model (2015) in Brian2 was extended to integrate automated environment initialization (including self-recovering installation triggers), automated MNIST dataset acquisition, and a robust checkpoint saving mechanism, while preserving the core architecture of the original model.
*   To mitigate the risk of losing checkpoint files or final training data stored within the volatile Colab environment, users are advised to integrate automated procedures for uploading outputs to cloud-based repositories or transferring them to local storage.






In [ ]:
# @title ## ***install & import brian2 (A session restart is required)***

import os
import subprocess
import sys

def check_brian2_installed():
    # Use pip show to check if brian2 is installed in the system environment
    result = subprocess.run([sys.executable, "-m", "pip", "show", "brian2"], capture_output=True)
    return result.returncode == 0

if check_brian2_installed():
    try:
        import brian2
        print("✅ Brian2 library is already installed and imported successfully! No download or restart needed.")
    except ImportError:
        print("⚠️ Brian2 is installed but requires a session restart to import. Restarting session now...")
        os.kill(os.getpid(), 9)
else:
    print("❌ Brian2 library not found. Downloading and installing...")
    !pip install brian2
    print("\n[INFO] First-time installation completed! Automatically restarting environment to apply updates...")
    os.kill(os.getpid(), 9)

In [ ]:
# @title ## ***Model & Function (If you haven't training data yet, set the "MODE" to "train" initailly.)***

from brian2 import *
from collections import defaultdict
from pathlib import Path
from progressbar import progressbar
from random import randrange, seed as rseed
from struct import unpack
import numpy as np

# Switch between "train", "observe", and "test" to tune parameters,
# observe excitatory spiking, and test accuracy, respectively.
# Use "plot" to plot the confusion matrix.
MODE = 'train'  # @param ["train", "observe", "test", "plot"] {type:"string"}

# Number of training, observation, and testing samples
N_TRAIN = 25_000

N_OBSERVE = 2_000
N_TEST = 1_000

# Random seed value
SEED = 42

# Storage paths
MNIST_PATH = Path('mnist')
DATA_PATH = Path('data')

# Number of weight save points
N_SAVE_POINTS = 100

# Don't change these values unless you know what you're doing.

N_INP = 784
N_NEURONS = 400
V_EXC_REST = -65 * mV
V_INH_REST = -60 * mV
INTENSITY = 2

# Weights of exc->inh and inh->exc synapses
W_EXC_INH = 10.4
W_INH_EXC = 17.0

def download_mnist():
    import urllib.request
    import gzip
    import shutil

    MNIST_PATH.mkdir(parents=True, exist_ok=True)
    base_url = 'https://ossci-datasets.s3.amazonaws.com/mnist/'
    files = [
        'train-images-idx3-ubyte.gz',
        'train-labels-idx1-ubyte.gz',
        't10k-images-idx3-ubyte.gz',
        't10k-labels-idx1-ubyte.gz'
    ]
    for f in files:
        dest_file = MNIST_PATH / f.replace('.gz', '')
        if dest_file.exists():
            continue
        gz_file = MNIST_PATH / f
        print(f"Downloading {f}...")
        try:
            urllib.request.urlretrieve(base_url + f, gz_file)
        except Exception as e:
            # Fallback to secondary mirror if S3 is slow or fails
            mirror_url = 'https://raw.githubusercontent.com/gndg/mnist-files/master/'
            urllib.request.urlretrieve(mirror_url + f, gz_file)
        print(f"Extracting {f}...")
        with gzip.open(gz_file, 'rb') as f_in:
            with open(dest_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        gz_file.unlink()
    X_train, Y_train = read_mnist(training=True)
    X_test, Y_test = read_mnist(training=False)
    print(f"Trainning Set: {X_train.shape[0]} Samples")
    print(f"Testing Set: {X_test.shape[0]} Samples")

def save_npy(arr, path):
    arr = np.array(arr)
    print('%-9s %-15s => %-30s' % ('Saving', arr.shape, path))
    np.save(path, arr)

def load_npy(path):
    arr = np.load(path)
    print('%-9s %-30s => %-15s' % ('Loading', path, arr.shape))
    return arr

def read_mnist(training):
    tag = 'train' if training else 't10k'
    images = open(MNIST_PATH / ('%s-images-idx3-ubyte' % tag), 'rb')
    images.read(4)
    n_images = unpack('>I', images.read(4))[0]
    n_rows = unpack(">I", images.read(4))[0]
    n_cols = unpack(">I", images.read(4))[0]

    labels = open(MNIST_PATH / ('%s-labels-idx1-ubyte' % tag), 'rb')
    labels.read(8)  # Changed from 4 to 8 to skip 'magic number' and 'number of items' fields

    x = np.frombuffer(images.read(), dtype = np.uint8)
    x = x.reshape(n_images, -1) / 8.0
    y = np.frombuffer(labels.read(), dtype = np.uint8)
    return x, y

def build_network(training, initial_weights=None, initial_theta=None):
    eqs = '''
    dv/dt = (v_rest - v + i_exc + i_inh) / tau_mem  : volt (unless refractory)
    i_exc = ge * -v                         : volt
    i_inh = gi * (v_inh_base - v)           : volt
    dge/dt = -ge/(1 * ms)                   : 1
    dgi/dt = -gi/(2 * ms)                   : 1
    dtimer/dt = 1                           : second
    '''
    reset = 'v = %r; timer = 0 * ms' % V_EXC_REST
    if training:
        exc_eqs = eqs + '''
        dtheta/dt = -theta / (1e7 * ms)         : volt
        '''
        if initial_theta is not None:
            arr_theta = initial_theta
        else:
            arr_theta = np.ones(N_NEURONS) * 20 * mV
        reset += '; theta += 0.05 * mV'
    else:
        exc_eqs = eqs + '''
        theta                                   : volt
        '''
        arr_theta = load_npy(DATA_PATH / 'theta.npy') * volt
    exc_eqs = Equations(exc_eqs,
                        tau_mem = 100 * ms,
                        v_rest = V_EXC_REST,
                        v_inh_base = -100 * mV)
    # Note that this neuron has a bit of un unusual refractoriness mechanism:
    # The membrane potential is clamped for 5ms, but spikes are prevented for 50ms
    # This has been taken from the original code.
    ng_exc = NeuronGroup(
        N_NEURONS, exc_eqs,
        threshold = 'v > (theta - 72 * mV) and (timer > 50 * ms)',
        refractory = 5 * ms,
        reset = reset,
        method = 'euler',
        name = 'exc')
    ng_exc.v = V_EXC_REST
    ng_exc.theta = arr_theta

    inh_eqs = Equations(eqs,
                        tau_mem = 10 * ms,
                        v_rest = V_INH_REST,
                        v_inh_base = -85 * mV)
    ng_inh = NeuronGroup(N_NEURONS, inh_eqs,
                         threshold = 'v > -40 * mV',
                         refractory = 2 * ms,
                         reset = 'v = -45 * mV',
                         method = 'euler',
                         name = 'inh')
    ng_inh.v = V_INH_REST

    syns_exc_inh = Synapses(ng_exc, ng_inh,
                            on_pre = 'ge_post += %f' % W_EXC_INH)
    syns_exc_inh.connect(j = 'i')

    syns_inh_exc = Synapses(ng_inh, ng_exc,
                            on_pre = 'gi_post += %f' % W_INH_EXC)
    syns_inh_exc.connect("i != j")

    pg_inp = PoissonGroup(N_INP, 0 * Hz, name = 'inp')

    # During training, inp->exc synapse weights are plastic.
    model = 'w : 1'
    on_post = ''
    on_pre = 'ge_post += w'
    if training:
        on_pre += '; pre = 1.; w = clip(w - 0.0001 * post1, 0, 1.0)'
        on_post += 'post2bef = post2; w = clip(w + 0.01 * pre * post2bef, 0, 1.0); post1 = 1.; post2 = 1.'
        model += '''
        post2bef                        : 1
        dpre/dt   = -pre/(20 * ms)      : 1 (event-driven)
        dpost1/dt = -post1/(20 * ms)    : 1 (event-driven)
        dpost2/dt = -post2/(40 * ms)    : 1 (event-driven)
        '''
        if initial_weights is not None:
            weights = initial_weights
        else:
            weights = (np.random.random(N_INP * N_NEURONS) + 0.01) * 0.3
    else:
        weights = load_npy(DATA_PATH / 'weights.npy')

    syns_inp_exc = Synapses(
        pg_inp, ng_exc,
        model = model,
        on_pre = on_pre,
        on_post = on_post,
        name = 'inp_exc'
    )
    syns_inp_exc.connect(True)
    syns_inp_exc.delay = 'rand() * 10 * ms'
    syns_inp_exc.w = weights

    exc_mon = SpikeMonitor(ng_exc, name = 'sp_exc')
    net = Network([pg_inp, ng_exc, ng_inh,
                   syns_inp_exc, syns_exc_inh, syns_inh_exc,
                   exc_mon])
    # Initialize
    net.run(0 * ms)
    return net

def show_sample(net, sample, intensity):
    exc_mon = net['sp_exc']
    prev = exc_mon.count[:]
    net['inp'].rates = sample * intensity * Hz
    net.run(350 * ms)
    # Don't count spikes occuring during the 150 ms rest.
    next = exc_mon.count[:]
    net['inp'].rates = 0 * Hz
    net.run(150 * ms)
    pat = next - prev
    cnt = np.sum(pat)
    if cnt < 5:
        return show_sample(net, sample, intensity + 1)
    return pat

def predict(groups, rates):
    return np.argmax([rates[grp].mean() for grp in groups])

def test():
    download_mnist()

    conf = np.zeros((10, 10))
    assign = np.load(DATA_PATH / 'assign.npy')
    groups = [np.where(assign == i)[0] for i in range(10)]

    X, Y = read_mnist(False)
    net = build_network(False)
    for i in progressbar(range(N_TEST)):
        ix = randrange(len(X))
        exc = show_sample(net, X[ix], INTENSITY)
        guess = predict(groups, exc)
        real = Y[ix]
        conf[real, guess] += 1

    print('Accuracy: %6.3f' % (np.trace(conf) / np.sum(conf)))
    conf = conf/conf.sum(axis=1)[:,None]
    print(np.around(conf, 2))
    save_npy(conf, DATA_PATH / 'confusion.npy')

def normalize_plastic_weights(syns):
    conns = np.reshape(syns.w, (N_INP, N_NEURONS))
    col_sums = np.sum(conns, axis = 0)
    factors = 78./ col_sums
    conns *= factors
    syns.w = conns.reshape(-1)

def stats(net):
    tick = defaultclock.timestep[:]
    cnt = np.sum(net['sp_exc'].count[:])

    inp_exc = net['inp_exc']
    w_mu = np.mean(inp_exc.w)
    w_std = np.std(inp_exc.w)

    exc = net['exc']
    theta = exc.theta / mV
    theta_mu = np.mean(theta)
    theta_sig = np.std(theta)
    return [tick, cnt, w_mu, w_std, theta_mu, theta_sig]


def train():
    download_mnist()

    X, Y = read_mnist(True)
    n_samples = X.shape[0]

    # Checkpoint configuration
    checkpoint_weights = DATA_PATH / 'checkpoint_weights.npy'
    checkpoint_theta = DATA_PATH / 'checkpoint_theta.npy'
    checkpoint_stats = DATA_PATH / 'checkpoint_stats.npy'
    checkpoint_w_hist = DATA_PATH / 'checkpoint_w_hist.npy'
    checkpoint_idx = DATA_PATH / 'checkpoint_idx.txt'

    start_idx = 0
    init_w = None
    init_theta = None
    rows = []
    w_hist = []

    # Check if a checkpoint exists
    if checkpoint_idx.exists() and checkpoint_weights.exists() and checkpoint_theta.exists():
        try:
            start_idx = int(checkpoint_idx.read_text().strip())
            print(f"Found checkpoint! Resuming training from sample {start_idx}...")
            init_w = np.load(checkpoint_weights)
            init_theta = np.load(checkpoint_theta) * volt
            if checkpoint_stats.exists():
                rows = list(np.load(checkpoint_stats))
            if checkpoint_w_hist.exists():
                w_hist = list(np.load(checkpoint_w_hist))
        except Exception as e:
            print(f"Failed to load checkpoint ({e}). Starting from scratch...")
            start_idx = 0
            rows = []
            w_hist = []

    net = build_network(True, initial_weights=init_w, initial_theta=init_theta)

    if start_idx == 0:
        rows = [stats(net) + [-1]]
        w_hist = [np.array(net['inp_exc'].w)]

    ratio = max(N_TRAIN // N_SAVE_POINTS, 1)

    for i in progressbar(range(start_idx, N_TRAIN)):
        ix = i % n_samples
        normalize_plastic_weights(net['inp_exc'])
        show_sample(net, X[ix], INTENSITY)
        rows.append(stats(net) + [Y[ix]])
        if i % ratio == 0:
            w_hist.append(np.array(net['inp_exc'].w))

        # Save checkpoint periodically (every 100 samples)
        if i > start_idx and i % 100 == 0:
            checkpoint_idx.write_text(str(i))
            np.save(checkpoint_weights, np.array(net['inp_exc'].w))
            np.save(checkpoint_theta, np.array(net['exc'].theta / volt))
            np.save(checkpoint_stats, np.array(rows))
            np.save(checkpoint_w_hist, np.array(w_hist))
            print(f"Checkpoint Saved!")

    save_npy(rows, DATA_PATH / 'train_stats.npy')
    save_npy(w_hist, DATA_PATH / 'train_w_hist.npy')
    save_npy(net['inp_exc'].w, DATA_PATH / 'weights.npy')
    save_npy(net['exc'].theta, DATA_PATH / 'theta.npy')

    # Clean up checkpoint files on successful completion
    for f in [checkpoint_weights, checkpoint_theta, checkpoint_stats, checkpoint_w_hist, checkpoint_idx]:
        if f.exists():
            f.unlink()
    for f in DATA_PATH.glob('checkpoint_*.part*'):
        if f.exists():
            f.unlink()

def observe():
    download_mnist()

    X, Y = read_mnist(True)
    n_samples = X.shape[0]
    net = build_network(False)

    # Observe Checkpoint configuration
    chk_idx_path = DATA_PATH / 'observe_checkpoint_idx.txt'
    chk_responses_path = DATA_PATH / 'observe_checkpoint_responses.npy'
    chk_stats_path = DATA_PATH / 'observe_checkpoint_stats.npy'

    start_idx = 0
    rows = []
    responses = defaultdict(list)

    # Try to load existing checkpoint for observe
    if chk_idx_path.exists() and chk_responses_path.exists() and chk_stats_path.exists():
        try:
            start_idx = int(chk_idx_path.read_text().strip())
            print(f"Found observe checkpoint! Resuming from sample {start_idx}...")
            rows = list(np.load(chk_stats_path))
            # Load dict from numpy structured array
            loaded_dict = np.load(chk_responses_path, allow_pickle=True).item()
            responses = defaultdict(list, loaded_dict)
        except Exception as e:
            print(f"Failed to load observe checkpoint ({e}). Starting from scratch...")
            start_idx = 0
            rows = []
            responses = defaultdict(list)

    if start_idx == 0:
        rows = [stats(net) + [-1]]

    for i in progressbar(range(N_OBSERVE)):
        ix = i % n_samples
        sample = X[ix]
        cls = Y[ix]
        exc = show_sample(net, sample, INTENSITY)
        rows.append(stats(net) + [Y[ix]])
        responses[cls].append(exc)

        # Save observe checkpoint every 100 samples
        if i > start_idx and i % 100 == 0:
            chk_idx_path.write_text(str(i))
            np.save(chk_stats_path, np.array(rows))
            # Convert defaultdict to standard dict to allow numpy pickle saving
            np.save(chk_responses_path, dict(responses))
            print(f"Observe Checkpoint Saved at sample {i}!")

    res = np.zeros((10, N_NEURONS))
    for cls, vals in responses.items():
        res[cls] = np.array(vals).mean(axis = 0)

    assign = np.argmax(res, axis = 0)
    save_npy(assign, DATA_PATH / 'assign.npy')
    save_npy(rows, DATA_PATH / 'observe_stats.npy')

    # Clean up observe checkpoint files on successful completion
    for f in [chk_idx_path, chk_responses_path, chk_stats_path]:
        if f.exists():
            f.unlink()
    for f in DATA_PATH.glob('observe_checkpoint_*.part*'):
        if f.exists():
            f.unlink()

def plot():
    import itertools
    conf = np.load(DATA_PATH / "confusion.npy")

    import matplotlib.pyplot as plt

    plt.imshow(100*conf, interpolation="nearest", cmap=plt.cm.Blues)
    for i, j in itertools.product(range(conf.shape[0]), range(conf.shape[1])):
        if conf[i, j] == 0:
            continue
        plt.text(
            j,
            i,
            f"{round(100*conf[i, j])}%",
            horizontalalignment="center",
            verticalalignment="center",
            color="white" if conf[i, j] > 0.5 else "black",
        )
    plt.colorbar()
    plt.xticks(range(10))
    plt.yticks(range(10))
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.show()


In [ ]:
# @title ## ***Conduct***

if __name__ == '__main__':
    seed(SEED)
    rseed(SEED)
    DATA_PATH.mkdir(parents = True, exist_ok = True)
    cmds = dict(train=train, observe=observe, test=test, plot=plot)
    cmds[MODE]()